# Notebook 1 — Dataset Construction

Builds the instruction-tuning dataset from seed examples + GPT-4 synthetic generation.

**Input:** `data/raw/seed_examples.jsonl` (20 hand-crafted examples)

**Output:** `data/processed/train.jsonl` and `data/processed/test.jsonl`

In [ ]:
import sys, json
sys.path.append('../src')
from data_utils import load_jsonl, save_jsonl

seed = load_jsonl('../data/raw/seed_examples.jsonl')
print(f'{len(seed)} seed examples loaded')
print(json.dumps(seed[0], indent=2))

## Step 2 — Validate seed examples

In [ ]:
from data_utils import is_valid_policy

errors = []
for i, ex in enumerate(seed):
    if not all(k in ex for k in ['instruction', 'input', 'output']):
        errors.append(f'Example {i}: missing required keys')
    elif not is_valid_policy(ex['output'].get('policy', {})):
        errors.append(f'Example {i}: invalid policy structure')

if errors:
    for e in errors: print(f'ERROR: {e}')
else:
    print('All seed examples valid')

## Step 3 — Synthetic generation with GPT-4

Uses seed examples as few-shot context. Review all outputs before saving.

Requires `OPENAI_API_KEY` environment variable.

In [ ]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])

FEW_SHOT = json.dumps(seed[:3], indent=2)

SYSTEM_PROMPT = """You generate AWS IAM policy training examples.
Each example must be a valid JSON object with keys: instruction, input, output.
The output must contain: policy (valid AWS IAM JSON), nist_controls (list), risk_notes (string).
Follow the exact same structure as the examples provided."""

def generate_synthetic_batch(n=10, service_hint=''):
    user_msg = f"""
Here are example training records:
{FEW_SHOT}

Generate {n} NEW examples following the same structure.
Focus on: {service_hint or 'varied AWS services (S3, EC2, RDS, Lambda, IAM, KMS, SQS, SNS)'}.
Return a JSON array of {n} objects. No explanation, just the JSON array.
"""
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[{'role': 'system', 'content': SYSTEM_PROMPT},
                  {'role': 'user', 'content': user_msg}],
        temperature=0.8,
        response_format={'type': 'json_object'}
    )
    raw = json.loads(response.choices[0].message.content)
    # GPT-4 may wrap in a key — unwrap if needed
    if isinstance(raw, list):
        return raw
    return next(v for v in raw.values() if isinstance(v, list))

# Generate in batches — review each batch before proceeding
synthetic = generate_synthetic_batch(n=10, service_hint='ECS, CloudWatch, Secrets Manager')
print(f'{len(synthetic)} synthetic examples generated')
print(json.dumps(synthetic[0], indent=2))

## Step 4 — Manual review

Scroll through synthetic examples. Remove any with hallucinated ARNs, wildcard abuse, or incorrect NIST mappings.

In [ ]:
# Save synthetic to raw for manual review before merging
save_jsonl(synthetic, '../data/raw/synthetic_batch_01.jsonl')

# After review: load approved synthetic + seeds
approved_synthetic = load_jsonl('../data/raw/synthetic_batch_01.jsonl')  # edit file to remove bad examples
all_examples = seed + approved_synthetic
print(f'Total after merge: {len(all_examples)}')

## Step 5 — Train/test split and save

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(all_examples, test_size=0.1, random_state=42)

save_jsonl(train, '../data/processed/train.jsonl')
save_jsonl(test, '../data/processed/test.jsonl')

print(f'Train: {len(train)} | Test: {len(test)}')
print('Dataset ready — proceed to notebook 02_finetune.ipynb')